# 2. Transfer learning на EuroSAT: head-only vs full fine-tuning

## 1. Окружение и данные


In [ ]:
import random
from copy import deepcopy
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

SEED = 42
FAST_MODE = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device, "| FAST_MODE:", FAST_MODE)


ResNet18 обучалась на natural images, а EuroSAT содержит спутниковые снимки. Используем ImageNet-нормализацию и
resize 96×96 для скорости.


In [ ]:
DATA_DIR = Path("data")
IMAGE_SIZE = 96 if FAST_MODE else 160
IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
base_ds = datasets.EuroSAT(DATA_DIR, download=True)
train_source = datasets.EuroSAT(DATA_DIR, transform=train_transform)
val_source = datasets.EuroSAT(DATA_DIR, transform=eval_transform)
class_names = base_ds.classes; targets_np = np.asarray(base_ds.targets)
rng_split = np.random.default_rng(SEED); train_idx, val_idx = [], []
for class_id in range(len(class_names)):
    idx = np.flatnonzero(targets_np == class_id); rng_split.shuffle(idx)
    if FAST_MODE: train_idx.extend(idx[:500]); val_idx.extend(idx[500:650])
    else:
        cut = int(0.8 * len(idx)); train_idx.extend(idx[:cut]); val_idx.extend(idx[cut:])
train_ds, val_ds = Subset(train_source, train_idx), Subset(val_source, val_idx)
kwargs = dict(batch_size=128, num_workers=2, pin_memory=torch.cuda.is_available(), persistent_workers=True)
train_loader = DataLoader(train_ds, shuffle=True, **kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **kwargs)
print("classes:", class_names, "| train:", len(train_ds), "| val:", len(val_ds))


## 2. Загрузка pretrained backbone и замена head

`weights=...DEFAULT` загружает обученные на ImageNet веса. Старый `fc` решает задачу на 1000 классов;
заменяем его на слой для 10 классов EuroSAT. Новый слой инициализируется случайно.


In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
old_head = model.fc
model.fc = nn.Linear(old_head.in_features, 10)
model = model.to(device)

print("Старый head:", old_head)
print("Новый head:", model.fc)
print("Категорий исходных весов:", len(weights.meta["categories"]))
print("Официальная preprocessing recipe:", weights.transforms())


In [ ]:
def parameter_report(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"total={total:,} | trainable={trainable:,} ({100*trainable/total:.2f}%)")
    return total, trainable

def freeze_for_feature_extraction(model):
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.fc.parameters():
        parameter.requires_grad = True

freeze_for_feature_extraction(model)
parameter_report(model)
assert all(not p.requires_grad for name, p in model.named_parameters() if not name.startswith("fc."))
assert all(p.requires_grad for p in model.fc.parameters())


## 3. Feature extraction

Заморозить параметры недостаточно: `model.train()` всё равно обновляет running statistics BatchNorm.
В режиме feature extraction после `model.train()` переводим все BN обратно в `eval()`.


In [ ]:
def freeze_batchnorm_stats(model):
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()

def train_one_epoch(model, loader, optimizer, loss_fn, freeze_bn=False):
    model.train()
    if freeze_bn:
        freeze_batchnorm_stats(model)
    total_loss = total_correct = total = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = loss_fn(logits, targets)
        loss.backward()
        optimizer.step()
        total += targets.size(0)
        total_loss += loss.item() * targets.size(0)
        total_correct += (logits.argmax(1) == targets).sum().item()
    return total_loss / total, total_correct / total

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = total_correct = total = 0
    with torch.inference_mode():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = loss_fn(logits, targets)
            total += targets.size(0)
            total_loss += loss.item() * targets.size(0)
            total_correct += (logits.argmax(1) == targets).sum().item()
    return total_loss / total, total_correct / total


In [ ]:
loss_fn = nn.CrossEntropyLoss()
head_optimizer = torch.optim.AdamW(model.fc.parameters(), lr=3e-3, weight_decay=1e-4)
head_epochs = 2 if FAST_MODE else 5
history = []

for epoch in range(head_epochs):
    train_metrics = train_one_epoch(model, train_loader, head_optimizer, loss_fn, freeze_bn=True)
    val_metrics = evaluate(model, val_loader, loss_fn)
    history.append(("head", *train_metrics, *val_metrics))
    print(f"head {epoch+1:02d}/{head_epochs} | train={train_metrics[1]:.3f} | val={val_metrics[1]:.3f}")

feature_extractor_acc = history[-1][-1]
head_state = deepcopy(model.state_dict())


## 4. Full fine-tuning

Стартуем с результата head-only, размораживаем все параметры и продолжаем обучение.


In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = True
total, trainable = parameter_report(model)
assert total == trainable

head_ids = {id(parameter) for parameter in model.fc.parameters()}
backbone_params = [parameter for parameter in model.parameters() if id(parameter) not in head_ids]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 5e-5, "name": "backbone"},
    {"params": model.fc.parameters(), "lr": 5e-4, "name": "head"},
], weight_decay=1e-4)
for group in optimizer.param_groups:
    print(f"{group['name']:>8}: lr={group['lr']:.1e}, params={sum(p.numel() for p in group['params']):,}")


In [ ]:
fine_tune_epochs = 1 if FAST_MODE else 4
for epoch in range(fine_tune_epochs):
    train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, freeze_bn=True)
    val_metrics = evaluate(model, val_loader, loss_fn)
    history.append(("fine-tune", *train_metrics, *val_metrics))
    print(f"fine-tune {epoch+1:02d}/{fine_tune_epochs} | train={train_metrics[1]:.3f} | val={val_metrics[1]:.3f}")

fine_tuned_acc = history[-1][-1]
print(f"feature extraction: {feature_extractor_acc:.3f}")
print(f"after fine-tuning:  {fine_tuned_acc:.3f}")
print(f"delta:              {fine_tuned_acc-feature_extractor_acc:+.3f}")


In [ ]:
labels = [f"{stage}-{i+1}" for i, (stage, *_rest) in enumerate(history)]
train_acc = [row[2] for row in history]
val_acc = [row[4] for row in history]
x = np.arange(len(history))
plt.figure(figsize=(9, 4))
plt.plot(x, train_acc, "o-", label="train")
plt.plot(x, val_acc, "o-", label="val")
plt.xticks(x, labels, rotation=30, ha="right")
plt.ylim(0, 1); plt.grid(); plt.legend(); plt.ylabel("accuracy"); plt.tight_layout(); plt.show()


## 5. Сохранение воспроизводимой контрольной точки

Сохраняйте не объект модели целиком, а `state_dict` плюс минимальную конфигурацию и связь с weights/preprocessing.


In [ ]:
checkpoint = {
    "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "architecture": "torchvision.resnet18",
    "weights": str(weights),
    "num_classes": 10,
    "classes": class_names,
    "image_size": IMAGE_SIZE,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
    "seed": SEED,
    "fast_mode": FAST_MODE,
}
checkpoint_path = Path("resnet18_eurosat_finetuned.pt")
torch.save(checkpoint, checkpoint_path)
print("Сохранено:", checkpoint_path, f"({checkpoint_path.stat().st_size/1e6:.1f} MB)")

# Проверка загрузки в новую модель.
restored = resnet18(weights=None)
restored.fc = nn.Linear(restored.fc.in_features, checkpoint["num_classes"])
restored.load_state_dict(checkpoint["model_state"])
restored.eval()
print("Checkpoint успешно загружен")


## Обучение с нуля

Чтобы отделить пользу transfer learning от архитектуры, создайте `resnet18(weights=None)` и обучите все параметры
с более высоким LR и большим числом эпох.

## Практика

1. Напишите функцию, которая возвращает имена всех trainable parameters.
2. Разморозьте `layer3` и дайте ему LR в 3–10 раз меньше, чем `layer4`.
3. Сравните `IMAGE_SIZE=96` и `160`: время, память, accuracy.
4. Загрузите `head_state` и убедитесь, что получили модель до fine-tuning.
5. Замените EuroSAT собственным `ImageFolder` и сохраните `class_to_idx` в checkpoint.


## Мини-квиз

1. Чем feature extraction отличается от fine-tuning?  
2. Почему новый head нельзя заморозить вместе с backbone?  
3. Почему pretrained backbone обычно получает меньший LR?  
4. Почему `requires_grad=False` не замораживает статистики BatchNorm?  
5. Что обязано совпасть между обучением и inference?  
6. Что проверить, если fine-tuning хуже head-only baseline?


## Подробнее

- [PyTorch: Transfer Learning for Computer Vision](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [TorchVision: models and pretrained weights](https://docs.pytorch.org/vision/stable/models.html)
- [PyTorch: Saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)

